# 22DM015 Final Project — Financial PhraseBank
## Person D: Data prep + Part 2 augmentation/generation + Part 4 distillation

**Dataset:** `takala/financial_phrasebank`, config `sentences_allagree` (2,264 sentences).
**Labels:** 0 = negative, 1 = neutral, 2 = positive.

### Shared data contract (set by Person D — do NOT re-split)
- Splits are committed under `data/` as CSVs: `train` (1584), `val` (227), `test` (453), `labeled_32` (32).
- The **32 labelled** examples are a balanced sample from train (11 neg / 10 neu / 11 pos).
- Part 2 'unlabelled' data = train minus the 32 (`unlabeled_pool`).
- Evaluate headline numbers on **`test`** only; tune on **`val`**. Use `eval_utils.evaluate` so we're comparable.
- Log every result with `eval_utils.log_result(...)` into `results/results.csv`.

> **AI disclosure:** any AI-generated code/text in this notebook must be watermarked and declared (course rule). Interpretation, methodology justification, and analysis must be student-authored.

This notebook is the **source of truth for the data**. Running the build cell regenerates the CSVs under `data/` that A/B/C consume. Commit those CSVs so everyone is byte-identical.

In [ ]:
# --- Reproducibility seed (required by the assignment) ---
import os, random, sys
import numpy as np
SEED = 618
random.seed(SEED); np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# Make the shared helpers importable (they live in the repo root, one level up).
sys.path.append(os.path.abspath('..'))
import data_utils as du
import eval_utils as eu

splits = du.load_splits()            # identical data for everyone
train, val, test = splits['train'], splits['val'], splits['test']
labeled_32 = splits['labeled_32']
pool = du.unlabeled_pool(splits)     # Part 2 'unlabelled' data
PERSON = 'D'
for k, v in splits.items():
    print(f'{k:11s} {len(v):5d}', dict(v['label_name'].value_counts()))

## 0. Build the shared splits (run once, then commit `data/`)
Stratified 70/10/20 with SEED=618; balanced 32-shot (11 neg / 10 neu / 11 pos) from train. Logic lives in `data_utils.build_splits()` so it's auditable and reproducible.

In [ ]:
# Regenerate the canonical CSVs. Safe to re-run: deterministic given SEED.
splits = du.build_splits()
for k, v in splits.items():
    print(f'{k:11s} n={len(v):4d}', dict(v['label_name'].value_counts().reindex(du.LABEL_NAMES)))
# sanity checks
tr, te, va = set(splits['train'].id), set(splits['test'].id), set(splits['val'].id)
assert not (tr & te) and not (tr & va) and not (te & va), 'split leakage!'
assert set(splits['labeled_32'].id) <= tr, '32-shot must come from train'
print('OK: no leakage, 32-shot ⊂ train')

## Part 2b. Dataset Augmentation WITHOUT an LLM (1)
Automated augmentation of the 32 labelled examples — e.g. EDA (synonym replace, random insert/swap/delete) or back-translation. Output `data/augmented_32.csv` with the SAME schema (id,text,label,label_name) for Person B to train on. Keep label consistency.
_Name the tradeoff: back-translation preserves meaning better; EDA is cheaper but noisier._

In [ ]:
# TODO: implement augmentation (e.g. nlpaug / nltk wordnet / MarianMT back-translation).
# Keep new rows labelled with the source row's label. Give synthetic ids (e.g. 100000+).
# aug.to_csv('../data/augmented_32.csv', index=False)

## Part 2d. Data Generation WITH an LLM (1)
Prompt an LLM to generate new labelled financial sentences (balanced across classes). Save `data/llm_generated.csv` (same schema). Declare the model + watermark. Person B trains on 32 + generated and reports the metric delta.

In [ ]:
# TODO: LLM generation. Validate generated labels, dedupe vs train/test to avoid leakage.
# gen.to_csv('../data/llm_generated.csv', index=False)
# Guard against leakage:
# assert gen['text'].isin(set(test['text'])).sum() == 0

## Part 4. Model Distillation / Quantization (3) — start thinking
Take the best model (likely Person B's fine-tuned BERT) and produce a lighter student.
- **4a (1.5):** distillation (e.g. DistilBERT student via logit/feature distillation) and/or quantization (dynamic int8 with `torch.quantization`, or bitsandbytes). Document tools.
- **4b (0.5):** compare student vs teacher on `test` — macro-F1 *and* inference speed / size.
- **4c (1):** analyse where the student degrades; propose improvements.
Reuse `eu.evaluate` + `eu.log_result('distilbert-student','distilled',...)` so it lands in the same table.

In [ ]:
# Placeholder for Part 4 — depends on Person B's trained teacher checkpoint.
# Plan: load teacher, distill to DistilBERT student, then time inference and compare size.
print('Part 4 scaffold — fill in once a teacher checkpoint exists')